# check chirality

In [ ]:

import MDAnalysis as mda
from rdkit import Chem


#unbond (resn HDA+HDC+HDG+HDT and name O2P), (resn HDA+HDC+HDG+HDT and name C11L)
lst=[7, 9, 18, 20, 28, 30, 39, 41, 49, 51, 60, 62, 70, 72, 81, 83, 91, 93, 102, 104, 112, 114, 123, 125, 133, 135, 144, 146]
modified = sorted(lst)


print(f"The number of modified residues: {len(modified)}")

S = []
R = []
U = []   # unassigned / undefined chirality

with open("check_chirality2.csv", "w") as f:

    for i in modified:
        path = f"mol2/residue_{i}_with_link.mol2"

        mol = Chem.MolFromMol2File(path, sanitize=False, removeHs=False)
        if mol is None:
            line = f"Residue {i} | ERROR: cannot read mol2\n"
            U.append(line)
            continue

        Chem.AssignAtomChiralTagsFromStructure(
            mol, replaceExistingTags=True
        )

        chiral_centers = Chem.FindMolChiralCenters(
            mol, includeUnassigned=True
        )

        found_P = False

        for idx, tag in chiral_centers:
            atom = mol.GetAtomWithIdx(idx)
            if atom.GetSymbol() == 'P':
                found_P = True
                line = f"Residue {i} | P | chirality {tag}\n"

                if tag == 'S':
                    S.append(line)
                elif tag == 'R':
                    R.append(line)
                else:
                    U.append(line)

        if not found_P:
            U.append(f"Residue {i} | P | chirality NONE\n")

    f.write("=== S chirality ===\n")
    f.writelines(S)

    f.write("\n=== R chirality ===\n")
    f.writelines(R)

    f.write("\n=== Unassigned / missing chirality ===\n")
    f.writelines(U)

print("Done.\n")

# Print R residues explicitly
for line in R:
    words = line.strip().split()
    print(f"residue number for R : {words[1]}")

for line in U:
    words = line.strip().split()
    print(f"residue number for U : {words[1]}")
    
print("\nSummary:")
print("S residues:", len(S))
print("R residues:", len(R))
print("Unassigned residues:", len(U))
print("Total counted:", len(S) + len(R) + len(U))
